# LumenY — Sustained Move Model (Binary 4H Direction)

**Approach:** Predict the direction of the next 4 hours of cumulative price movement.

**Target:** `label_4H` — cumulative log return over the next 4 hours.
- If return > 0 → **UP** (1)
- If return <= 0 → **DOWN** (0)

**Single model:** LightGBM binary classifier (UP / DOWN), all 7 pairs pooled.

**Trading logic:** Enter on UP/DOWN signal with high confidence. Hold while model confirms same direction each hour. Exit when model flips or confidence drops. This naturally captures moves of variable length.

**Key difference from V1:** V1 predicts 1H direction. This model predicts 4H direction — should be more robust to single-candle noise.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report

FEATURES_DIR = Path('../backend/data/features')
MODELS_DIR   = Path('../backend/models_4/sustained_move')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── CRITICAL: Training cutoff ──
TRAIN_END = '2024-06-30'

# ── Target horizon ──
TARGET_COL = 'label_4H'  # cumulative log return over next 4 hours

print('Sustained Move Model — Binary 4H Direction')
print(f'Training cutoff: {TRAIN_END}')
print(f'Target: {TARGET_COL} → UP (1) if return > 0, DOWN (0) otherwise')

## 1. Load Data & Build Target

In [ ]:
df = pd.read_parquet(FEATURES_DIR / 'all_pairs_features_labels.parquet')

label_cols   = [c for c in df.columns if c.startswith('label_')]
drop_cols    = label_cols + ['pair']
feature_cols = [c for c in df.columns if c not in drop_cols]

print(f'Dataset: {df.shape}  ({df.index.min().date()} to {df.index.max().date()})')
print(f'Features: {len(feature_cols)}')
print(f'Label columns: {label_cols}')
print(f'Pairs: {sorted(df["pair"].unique())}')

# Split
df_train = df[df.index <= TRAIN_END].copy()
df_test  = df[df.index > TRAIN_END].copy()

# Remove rows where target is NaN
valid_train = df_train[TARGET_COL].notna()
valid_test  = df_test[TARGET_COL].notna()
df_train = df_train[valid_train]
df_test  = df_test[valid_test]

print(f'\nTrain: {len(df_train):,} rows  ({df_train.index.min().date()} to {df_train.index.max().date()})')
print(f'Test:  {len(df_test):,} rows  ({df_test.index.min().date()} to {df_test.index.max().date()})')

# ── Binary target: UP=1 if 4H return > 0, DOWN=0 otherwise ──
df_train['target'] = (df_train[TARGET_COL] > 0).astype(int)
df_test['target']  = (df_test[TARGET_COL] > 0).astype(int)

print(f'\n4H return stats (train):')
print(df_train[TARGET_COL].describe())

# Class distribution
for name, d in [('Train', df_train), ('Test', df_test)]:
    up_pct = d['target'].mean()
    print(f'\n{name}: UP={up_pct:.1%}, DOWN={1-up_pct:.1%}')

## 2. Walk-Forward Cross-Validation

In [ ]:
def walk_forward_splits(n, n_splits=5, test_ratio=0.1):
    """Expanding window walk-forward splits. All within training data."""
    test_size = int(n * test_ratio)
    splits = []
    for i in range(n_splits):
        test_start = int(n * 0.5) + i * (int(n * 0.5) // n_splits)
        test_end   = test_start + test_size
        if test_end > n:
            break
        splits.append((list(range(0, test_start)), list(range(test_start, test_end))))
    return splits

X_train_all = df_train[feature_cols].ffill().fillna(0)
y_train_all = df_train['target']

splits = walk_forward_splits(len(df_train))
print(f'Walk-forward splits: {len(splits)} (all within training set, before {TRAIN_END})')
for i, (train_idx, test_idx) in enumerate(splits):
    print(f'  Fold {i+1}: Train to {df_train.index[train_idx[-1]].date()} ({len(train_idx):,}) | '
          f'Test {df_train.index[test_idx[0]].date()} to {df_train.index[test_idx[-1]].date()} ({len(test_idx):,})')

In [ ]:
MODEL_PARAMS = {
    'objective':         'binary',
    'metric':            'binary_logloss',
    'boosting_type':     'gbdt',
    'n_estimators':      5000,
    'learning_rate':     0.02,
    'num_leaves':        64,
    'max_depth':         6,
    'min_child_samples': 50,
    'feature_fraction':  0.7,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'reg_alpha':         0.1,
    'reg_lambda':        0.1,
    'random_state':      42,
    'n_jobs':            -1,
    'verbose':           -1,
    'device':            'gpu',
}

fold_accs = []
fold_conf_accs = []  # accuracy on high-confidence predictions

for fold, (tr_idx, te_idx) in enumerate(splits):
    X_tr, y_tr = X_train_all.iloc[tr_idx], y_train_all.iloc[tr_idx]
    X_te, y_te = X_train_all.iloc[te_idx], y_train_all.iloc[te_idx]
    
    model = lgb.LGBMClassifier(**MODEL_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    
    probs = model.predict_proba(X_te)[:, 1]
    preds = (probs > 0.5).astype(int)
    acc = accuracy_score(y_te, preds)
    fold_accs.append(acc)
    
    # High-confidence accuracy (conf >= 0.55)
    conf = np.where(probs > 0.5, probs, 1 - probs)
    high_conf = conf >= 0.55
    if high_conf.sum() > 0:
        conf_acc = accuracy_score(y_te.values[high_conf], preds[high_conf])
        fold_conf_accs.append(conf_acc)
        n_conf = high_conf.sum()
    else:
        conf_acc = 0
        n_conf = 0
    
    print(f'  Fold {fold+1}: Acc={acc:.4f} ({acc*100:.1f}%) | '
          f'High-conf acc={conf_acc:.4f} ({conf_acc*100:.1f}%, n={n_conf:,}) | '
          f'best_iter={model.best_iteration_}')

print(f'\nCV Accuracy: {np.mean(fold_accs)*100:.1f}% +/- {np.std(fold_accs)*100:.1f}%')
if fold_conf_accs:
    print(f'CV High-Conf Accuracy: {np.mean(fold_conf_accs)*100:.1f}% +/- {np.std(fold_conf_accs)*100:.1f}%')

## 3. Train Final Model

In [ ]:
# ── Train final model with early stopping on held-out validation ──
VALID_RATIO = 0.1
n_train = int(len(X_train_all) * (1 - VALID_RATIO))

X_fit, y_fit = X_train_all.iloc[:n_train], y_train_all.iloc[:n_train]
X_val, y_val = X_train_all.iloc[n_train:], y_train_all.iloc[n_train:]

print(f'Training final model with early stopping...')
print(f'  Fit:  {len(X_fit):,} rows (up to {df_train.index[n_train-1].date()})')
print(f'  Val:  {len(X_val):,} rows ({df_train.index[n_train].date()} to {df_train.index[-1].date()})')

final_model = lgb.LGBMClassifier(**MODEL_PARAMS)
final_model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)])

print(f'\nBest iteration: {final_model.best_iteration_}')

# Save model bundle
bundle = {
    'model': final_model,
    'type': 'sustained_move_binary',
    'target': 'label_4H → UP/DOWN',
    'classes': {0: 'DOWN', 1: 'UP'},
    'feature_cols': feature_cols,
    'train_end': TRAIN_END,
    'cv_accuracy': np.mean(fold_accs),
    'n_iterations': final_model.best_iteration_,
}

save_path = MODELS_DIR / 'sustained_move_model.joblib'
joblib.dump(bundle, save_path)

print(f'\nSaved: {save_path}')
print(f'Model size: {save_path.stat().st_size / 1024 / 1024:.1f} MB')

## 4. Test Set — Raw Model Predictions

Evaluate the binary UP/DOWN predictions on the unseen test set.

In [ ]:
X_test = df_test[feature_cols].ffill().fillna(0)
y_test = df_test['target']
actual_return_4h = df_test[TARGET_COL]
actual_return_1h = df_test['label_1H']

# Raw predictions
probs_test = final_model.predict_proba(X_test)[:, 1]  # P(UP)
preds_test = (probs_test > 0.5).astype(int)

print(f'Test set: {len(X_test):,} rows ({df_test.index.min().date()} to {df_test.index.max().date()})')
print(f'\nBinary accuracy: {accuracy_score(y_test, preds_test)*100:.1f}%')
print(f'\n{classification_report(y_test, preds_test, target_names=["DOWN", "UP"])}')

# Prediction distribution
n_up = preds_test.sum()
n_down = len(preds_test) - n_up
print(f'Prediction distribution: UP={n_up:,} ({n_up/len(preds_test)*100:.1f}%), DOWN={n_down:,} ({n_down/len(preds_test)*100:.1f}%)')

# Directional accuracy on 4H return
pred_dir = np.where(probs_test > 0.5, 1, -1)
actual_dir_4h = np.sign(actual_return_4h.values)
dir_acc = (pred_dir == actual_dir_4h).mean()
print(f'\n4H directional accuracy: {dir_acc*100:.1f}%')

## 5. Confidence Analysis

Confidence = max(P(UP), P(DOWN)). Higher confidence should mean higher accuracy.

In [ ]:
signal_dir  = np.where(probs_test > 0.5, 1, -1)   # +1=UP, -1=DOWN
signal_conf = np.where(probs_test > 0.5, probs_test, 1 - probs_test)  # confidence
actual_dir  = np.sign(actual_return_1h.values)    # actual 1H direction for PnL

results = pd.DataFrame({
    'signal_dir': signal_dir,
    'signal_conf': signal_conf,
    'p_up': probs_test,
    'pred_class': preds_test,
    'actual_class': y_test.values,
    'actual_return_4h': actual_return_4h.values,
    'actual_return_1h': actual_return_1h.values,
    'actual_dir_1h': actual_dir,
    'pair': df_test['pair'].values,
}, index=df_test.index)

# ── Accuracy by confidence level ──
conf_thresholds = [0.50, 0.52, 0.54, 0.56, 0.58, 0.60, 0.65, 0.70]

print(f'{"Conf >=":<10} {"Signals":>8} {"% of total":>10} {"Dir Acc (4H)":>14} {"Dir Acc (1H)":>14} {"Avg |4H ret|":>14}')
print('-' * 75)

for ct in conf_thresholds:
    mask = signal_conf >= ct
    if mask.sum() < 50:
        continue
    
    sub = results[mask]
    actual_dir_4h = np.sign(sub['actual_return_4h'].values)
    dir_acc_4h = (sub['signal_dir'].values == actual_dir_4h).mean()
    dir_acc_1h = (sub['signal_dir'].values == sub['actual_dir_1h'].values).mean()
    avg_abs_ret = sub['actual_return_4h'].abs().mean()
    
    print(f'{ct:<10.2f} {mask.sum():>8,} {mask.sum()/len(results)*100:>9.1f}% {dir_acc_4h*100:>13.1f}% {dir_acc_1h*100:>13.1f}% {avg_abs_ret:>14.6f}')

## 6. Position Simulation — Hold Until Flip

Simulate realistic trading: enter on strong signal, hold while model confirms, exit when direction flips or confidence drops. PnL is computed on 1H returns (what you actually earn each hour you hold).

In [ ]:
def simulate_positions(results_df, entry_conf=0.45, hold_conf=0.35, max_hold=8, cooldown=1):
    """
    Simulate hold-until-flip trading.
    
    Entry: signal_conf >= entry_conf
    Hold:  signal still same direction and signal_conf >= hold_conf
    Exit:  direction flips, confidence drops below hold_conf, or max_hold reached
    Cooldown: wait N bars after exit before allowing new entry
    
    PnL: position_dir × actual_1H_return each hour we hold.
    """
    trades = []
    
    for pair in sorted(results_df['pair'].unique()):
        pair_data = results_df[results_df['pair'] == pair].sort_index()
        
        position = None
        cooldown_remaining = 0
        
        for i in range(len(pair_data)):
            row = pair_data.iloc[i]
            ts = pair_data.index[i]
            
            if position is not None:
                position['bars_held'] += 1
                hour_pnl = position['dir'] * row['actual_return_1h']
                position['pnl'] += hour_pnl
                
                same_dir = (row['signal_dir'] == position['dir'])
                conf_ok  = (row['signal_conf'] >= hold_conf)
                max_reached = (position['bars_held'] >= max_hold)
                
                if not same_dir or not conf_ok or max_reached:
                    trades.append({
                        'pair': pair,
                        'direction': 'UP' if position['dir'] == 1 else 'DOWN',
                        'entry_time': position['entry_time'],
                        'exit_time': ts,
                        'duration_h': position['bars_held'],
                        'pnl': position['pnl'],
                        'entry_conf': position['entry_conf'],
                        'exit_reason': 'max_hold' if max_reached else ('flip' if not same_dir else 'low_conf'),
                    })
                    position = None
                    cooldown_remaining = cooldown
            else:
                if cooldown_remaining > 0:
                    cooldown_remaining -= 1
                    continue
                
                if row['signal_conf'] >= entry_conf:
                    position = {
                        'dir': row['signal_dir'],
                        'entry_time': ts,
                        'entry_conf': row['signal_conf'],
                        'bars_held': 0,
                        'pnl': 0.0,
                    }
    
    return pd.DataFrame(trades)


# ── Run simulation with different entry confidence thresholds ──
print(f'{"Entry Conf":<12} {"Trades":>8} {"Avg Dur":>8} {"Win%":>8} {"Avg PnL":>12} {"Total PnL":>12} {"Sharpe":>8}')
print('-' * 75)

best_sharpe = 0
best_conf = 0.50

for ec in [0.50, 0.52, 0.54, 0.56, 0.58, 0.60, 0.65, 0.70]:
    trades_df = simulate_positions(results, entry_conf=ec, hold_conf=0.50, max_hold=8, cooldown=1)
    
    if len(trades_df) < 20:
        continue
    
    win_rate = (trades_df['pnl'] > 0).mean()
    avg_pnl = trades_df['pnl'].mean()
    total_pnl = trades_df['pnl'].sum()
    avg_dur = trades_df['duration_h'].mean()
    sharpe = (trades_df['pnl'].mean() / trades_df['pnl'].std()) * np.sqrt(252 * 24 / avg_dur) if trades_df['pnl'].std() > 0 else 0
    
    if sharpe > best_sharpe:
        best_sharpe = sharpe
        best_conf = ec
    
    print(f'{ec:<12.2f} {len(trades_df):>8,} {avg_dur:>7.1f}h {win_rate:>7.1%} {avg_pnl:>12.6f} {total_pnl:>12.4f} {sharpe:>8.2f}')

print(f'\nBest Sharpe at entry_conf={best_conf:.2f}')

## 7. Detailed Analysis of Best Config

In [ ]:
# Use best config for detailed analysis
trades_best = simulate_positions(results, entry_conf=best_conf, hold_conf=0.33, max_hold=8, cooldown=1)

print(f'=== Best config: entry_conf={best_conf:.2f} ===')
print(f'Total trades: {len(trades_best):,}')
print(f'Win rate: {(trades_best["pnl"] > 0).mean()*100:.1f}%')
print(f'Avg duration: {trades_best["duration_h"].mean():.1f}h')
print(f'Total PnL: {trades_best["pnl"].sum():.4f}')
print(f'Avg PnL/trade: {trades_best["pnl"].mean():.6f}')

# Duration distribution
print(f'\nDuration distribution:')
dur_counts = trades_best['duration_h'].value_counts().sort_index()
for dur, count in dur_counts.items():
    pnl_at_dur = trades_best[trades_best['duration_h'] == dur]['pnl']
    print(f'  {dur}h: {count:>4} trades, win rate {(pnl_at_dur > 0).mean()*100:.0f}%, avg PnL {pnl_at_dur.mean():.6f}')

# Exit reason distribution
print(f'\nExit reasons:')
for reason, group in trades_best.groupby('exit_reason'):
    print(f'  {reason}: {len(group):>4} trades ({len(group)/len(trades_best)*100:.0f}%), avg PnL {group["pnl"].mean():.6f}')

# Per-pair breakdown
print(f'\nPer-pair breakdown:')
print(f'{"Pair":<10} {"Trades":>8} {"Win%":>8} {"Avg Dur":>8} {"Avg PnL":>12} {"Total PnL":>12}')
print('-' * 60)
for pair in sorted(trades_best['pair'].unique()):
    pt = trades_best[trades_best['pair'] == pair]
    print(f'{pair:<10} {len(pt):>8} {(pt["pnl"]>0).mean():>7.1%} {pt["duration_h"].mean():>7.1f}h {pt["pnl"].mean():>12.6f} {pt["pnl"].sum():>12.4f}')

# Direction breakdown
print(f'\nDirection breakdown:')
for d in ['UP', 'DOWN']:
    dt = trades_best[trades_best['direction'] == d]
    print(f'  {d}: {len(dt)} trades, win rate {(dt["pnl"]>0).mean()*100:.1f}%, avg PnL {dt["pnl"].mean():.6f}')

## 8. Equity Curves

In [ ]:
def build_hourly_pnl(results_df, entry_conf, hold_conf=0.50, max_hold=8, cooldown=1):
    """Build hour-by-hour PnL series from position simulation (for equity curves)."""
    hourly_pnl = pd.Series(0.0, index=results_df.index)
    
    for pair in sorted(results_df['pair'].unique()):
        pair_data = results_df[results_df['pair'] == pair].sort_index()
        pair_pnl = pd.Series(0.0, index=pair_data.index)
        
        position = None
        cooldown_remaining = 0
        
        for i in range(len(pair_data)):
            row = pair_data.iloc[i]
            ts = pair_data.index[i]
            
            if position is not None:
                position['bars_held'] += 1
                pair_pnl.iloc[i] = position['dir'] * row['actual_return_1h']
                
                same_dir = (row['signal_dir'] == position['dir'])
                conf_ok  = (row['signal_conf'] >= hold_conf)
                max_reached = (position['bars_held'] >= max_hold)
                
                if not same_dir or not conf_ok or max_reached:
                    position = None
                    cooldown_remaining = cooldown
            else:
                if cooldown_remaining > 0:
                    cooldown_remaining -= 1
                    continue
                if row['signal_conf'] >= entry_conf:
                    position = {
                        'dir': row['signal_dir'],
                        'entry_conf': row['signal_conf'],
                        'bars_held': 0,
                        'pnl': 0.0,
                    }
        
        hourly_pnl = hourly_pnl.add(pair_pnl, fill_value=0)
    
    return hourly_pnl


fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor('#080c14')

configs = [
    ('entry_conf=0.50', 0.50),
    ('entry_conf=0.54', 0.54),
    ('entry_conf=0.58', 0.58),
    (f'entry_conf={best_conf:.2f} (best)', best_conf),
]

# Deduplicate if best_conf is already in the list
seen = set()
unique_configs = []
for name, ec in configs:
    if ec not in seen:
        unique_configs.append((name, ec))
        seen.add(ec)
    if len(unique_configs) == 4:
        break

while len(unique_configs) < 4:
    unique_configs.append(('entry_conf=0.60', 0.60))

for ax, (name, ec) in zip(axes.flatten(), unique_configs):
    ax.set_facecolor('#080c14')
    
    hourly = build_hourly_pnl(results, entry_conf=ec)
    cum_pnl = hourly.cumsum()
    
    for pair in sorted(results['pair'].unique()):
        pair_hourly = build_hourly_pnl(results[results['pair'] == pair], entry_conf=ec)
        pair_cum = pair_hourly.cumsum()
        ax.plot(pair_cum.index, pair_cum.values, alpha=0.3, linewidth=0.7)
    
    ax.plot(cum_pnl.index, cum_pnl.values, color='#4fc3f7', linewidth=2, label='All pairs')
    ax.axhline(0, color=(1,1,1,0.2), linewidth=1, linestyle='--')
    
    trades_n = simulate_positions(results, entry_conf=ec, hold_conf=0.50, max_hold=8, cooldown=1)
    n_trades = len(trades_n)
    total = trades_n['pnl'].sum() if len(trades_n) > 0 else 0
    
    ax.set_title(f'{name} | {n_trades} trades, PnL={total:.2f}', color='white', fontsize=10)
    ax.tick_params(colors='white')
    ax.set_ylabel('Cumulative log return', color='white', fontsize=8)
    for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')

plt.suptitle('Sustained Move Model (Binary 4H) — Equity Curves (UNSEEN test set)\nHold-until-flip strategy, max_hold=8h, cooldown=1h', 
             color='white', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Comparison with V1 (Two-Stage 1H Model)

Compare sustained move model vs V1's best config (Top 10% vol + conf>=55%). Use the same 1H return PnL calculation for fair comparison.

In [ ]:
# ── Load V1 models for comparison ──
v1_vol_bundle = joblib.load(Path('../backend/models_4/vol_model.joblib'))
v1_dir_bundle = joblib.load(Path('../backend/models_4/dir_model.joblib'))
v1_vol_model = v1_vol_bundle['model']
v1_dir_model = v1_dir_bundle['model']
v1_vol_pct = v1_vol_bundle['vol_percentiles']
v1_feature_cols = v1_vol_bundle['feature_cols']

# V1 predictions on test set
X_test_v1 = df_test[v1_feature_cols].ffill().fillna(0)
v1_vol_preds = v1_vol_model.predict(X_test_v1)
v1_dir_probs = v1_dir_model.predict_proba(X_test_v1)[:, 1]  # P(UP)
v1_dir = np.where(v1_dir_probs > 0.5, 1, -1)
v1_conf = np.where(v1_dir_probs > 0.5, v1_dir_probs, 1 - v1_dir_probs)

v1_results = pd.DataFrame({
    'pred_vol': v1_vol_preds,
    'pred_dir': v1_dir,
    'confidence': v1_conf,
    'actual_return_1h': actual_return_1h.values,
    'pair': df_test['pair'].values,
}, index=df_test.index)

def v1_pnl(vol_pct_thresh, conf_thresh=0.50):
    mask = (v1_results['pred_vol'] > v1_vol_pct[vol_pct_thresh]) & (v1_results['confidence'] >= conf_thresh)
    if mask.sum() == 0:
        return 0, 0, 0, 0
    sub = v1_results[mask]
    pnl = sub['pred_dir'] * sub['actual_return_1h']
    acc = (np.sign(sub['actual_return_1h'].values) == sub['pred_dir'].values).mean()
    sharpe = (pnl.mean() / pnl.std()) * np.sqrt(252 * 24) if pnl.std() > 0 else 0
    return mask.sum(), acc, pnl.sum(), sharpe

print('=' * 80)
print('COMPARISON: Sustained Move (Binary 4H) vs V1 Two-Stage (1H)')
print('=' * 80)

print(f'\n{"Strategy":<45} {"Trades":>8} {"Acc/WinR":>10} {"Total PnL":>12} {"Sharpe":>8}')
print('-' * 85)

for vp, cf, label in [
    (50, 0.50, 'V1: Top50% vol'),
    (67, 0.50, 'V1: Top33% vol'),
    (90, 0.50, 'V1: Top10% vol'),
    (90, 0.55, 'V1: Top10% vol + conf>=55%'),
]:
    n, acc, total, sharpe = v1_pnl(vp, cf)
    print(f'{label:<45} {n:>8,} {acc:>9.1%} {total:>12.4f} {sharpe:>8.2f}')

print()

for ec in [0.50, 0.54, 0.58, 0.60, 0.65]:
    trades_df = simulate_positions(results, entry_conf=ec, hold_conf=0.50, max_hold=8, cooldown=1)
    if len(trades_df) == 0:
        continue
    win_r = (trades_df['pnl'] > 0).mean()
    total = trades_df['pnl'].sum()
    avg_dur = trades_df['duration_h'].mean()
    sharpe = (trades_df['pnl'].mean() / trades_df['pnl'].std()) * np.sqrt(252 * 24 / avg_dur) if trades_df['pnl'].std() > 0 else 0
    print(f'{"Sustained 4H: conf>=" + f"{ec:.2f}":<45} {len(trades_df):>8,} {win_r:>9.1%} {total:>12.4f} {sharpe:>8.2f}')

## 10. Feature Importance

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
fig.patch.set_facecolor('#080c14')
ax.set_facecolor('#080c14')

importance = pd.Series(final_model.feature_importances_, index=feature_cols)
importance = importance.sort_values(ascending=True).tail(30)

ax.barh(importance.index, importance.values, color='#4fc3f7', alpha=0.8)
ax.tick_params(colors='white', labelsize=8)
ax.set_title('Sustained Move Model — Top 30 Features', color='white', fontsize=12)
for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')

plt.tight_layout()
plt.show()

## 11. Summary

In [ ]:
print('=' * 60)
print('SUSTAINED MOVE MODEL (BINARY 4H) — TRAINING COMPLETE')
print('=' * 60)

print(f'\nSaved to: {MODELS_DIR.resolve()}')
for f in sorted(MODELS_DIR.glob('*.joblib')):
    print(f'  {f.name:<35} {f.stat().st_size / 1024 / 1024:.1f} MB')

print(f'\n-- Architecture --')
print(f'  Model: single LightGBM binary (UP/DOWN)')
print(f'  Target: label_4H (cumulative 4-hour log return > 0 → UP)')
print(f'  Features: {len(feature_cols)} (all features)')
print(f'  Training: early stopping on last 10% validation split')
print(f'  Final model: {final_model.best_iteration_} iterations')
print(f'  CV Accuracy: {np.mean(fold_accs)*100:.1f}% +/- {np.std(fold_accs)*100:.1f}%')

print(f'\n-- Trading Logic --')
print(f'  Entry: model predicts UP/DOWN with conf >= threshold')
print(f'  Hold: while model confirms same direction')
print(f'  Exit: direction flips, confidence drops, or max hold reached')
print(f'  PnL: position_dir x actual_1H_return each hour held')

print(f'\n-- Leakage Checklist --')
print(f'  [OK] TRAIN_END = {TRAIN_END} strict temporal split')
print(f'  [OK] Early stopping on temporal validation split (last 10% of train)')
print(f'  [OK] Walk-forward CV within training set only')
print(f'  [OK] No test-set statistics used in training')
print(f'  [OK] Target label_4H computed from future prices (proper forward label)')
print(f'  [OK] Position simulation is causal (only uses current-bar predictions)')

print(f'\n-- Test Period: {df_test.index.min().date()} to {df_test.index.max().date()} (UNSEEN) --')